# Colab: Bearing-UAV only / vis_seq_eval navigation
Colab에서 **런타임 → 런타임 유형 변경 → GPU**로 설정하고 **모두 실행**하세요.
`kagglehub.dataset_download("heeyunju/visloc-seq-eval")`로 데이터를 다운로드하고 반환된 폴더를 자동 사용합니다. 별도 데이터 업로드나 Kaggle GPU 실행은 필요하지 않습니다.
토큰 입력과 `whoami()` 인증 확인이 완료된 후 다운로드합니다. Kaggle Settings → API에서 발급한 토큰을 로그인 입력창에 넣으세요. 인증 후에도 403이면 해당 계정의 데이터 접근 권한과 동의 요구를 확인하세요.

마지막 셀에서 결과 ZIP을 다운로드합니다. `/content` 결과는 런타임 삭제 시 사라지므로 저장하세요. 맨 아래에 선택 사항인 Google Drive 결과 복사 셀도 있습니다.

Bearing-UAV 공개 cross_view 가중치만 준비합니다. 다른 모델은 설치하거나 실행하지 않습니다. 가중치는 자동 다운로드하며, 보유한 가중치를 쓰려면 준비 셀의 `BEARING_WEIGHTS_DIR`를 지정하세요.

짧은 검증(첫 구간 3장) 다음 전체 구간을 실행합니다. 각 단계는 별도 결과 폴더를 사용합니다. samples.csv의 map_x/map_y/dist_m, maps/<set>.jpg, <set>/k###_uav.jpg가 필요합니다. single_eval은 사용할 수 없습니다. ZIP 및 폴더 입력을 지원합니다.

실제 사진을 시간 순서로 재생하며 직전 추정 + 모의 VIO 이동량으로 다음 지도 블록을 선택합니다. GPS는 시작점·이동량 시뮬레이션·채점에만 사용합니다. 시작점은 모든 구간에서 정확히 고정하고 채점에서 제외합니다. 실제 VIO나 폐루프 비행 제어 성능은 아닙니다. Bearing에는 inlier가 없으므로 R0(예측을 그대로 사용)만 평가합니다. 128px=32m, 512px 지도 블록, 256px UAV, FP32 추론입니다.

결과: 걸음/구간 CSV, 10·20·32m 정확도, 마지막 40m 이내 비율, 오차 요약, 환경·가중치 해시·로그와 ZIP. 공개 pretrained의 학습 데이터 중복 여부는 별도 확인이 필요합니다.

## 1. 실행 설정

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, datetime, hashlib, shutil, traceback

WORK = Path('/content/bearing_sequence_nav')
WORK.mkdir(parents=True, exist_ok=True)
# 같은 런타임에서 설정 셀만 재실행하면 RUN을 유지합니다.
# 새 실험은 globals().pop('RUN', None) 실행 후 이 셀부터 다시 실행하세요.
if 'RUN' not in globals() or not Path(RUN).is_relative_to(WORK):
    RUN = WORK / ('run_' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    RUN.mkdir()
SOURCE = RUN/'source'; SOURCE.mkdir(exist_ok=True)
CACHE = WORK/'pretrained'; CACHE.mkdir(exist_ok=True)
WARMUP, REPEATS = 5, 1
METERS_PER_UNIT = 32.0
DATA_DIR = ''  # 다운로드한 데이터 안에 여러 seq_eval이 있을 때만 정확한 폴더 지정
SEED = 2026
ODO = 'vio'  # 'none' 또는 'vio': GPS 변위 기반 모의 이동 오차
SETS = []  # 비우면 모든 구간
print('결과 폴더:', RUN)

## 2. GPU 및 실행 도구

In [ ]:
import pandas as pd
import numpy as np
import torch
import cv2
from IPython.display import display, FileLink
if not torch.cuda.is_available():
    raise RuntimeError('Colab 런타임 유형을 GPU로 변경하세요.')
DEVICE = 'cuda:0'
MODELS = ['Bearing-UAV']

def save_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding='utf-8')

def hash_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024**2), b''): h.update(chunk)
    return h.hexdigest()

def make_env(name, packages):
    import sys
    import importlib.metadata as metadata
    env=WORK/'envs'/name
    py=env/'bin/python'
    env.parent.mkdir(parents=True,exist_ok=True)
    log=RUN/(name+'_environment.log')

    def command(args, required=True):
        result=subprocess.run([str(x) for x in args],stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT,text=True)
        output=result.stdout or ''
        with log.open('a',encoding='utf-8') as f:
            f.write(repr([str(x) for x in args])+chr(10)+output+chr(10))
        if output: print(output,flush=True)
        if required and result.returncode:
            raise RuntimeError(f'Environment command failed ({result.returncode}). See {log}: '+output[-4000:])
        return result

    def valid_python():
        if not py.exists(): return False
        try:
            probe=command([py,'-c',
                'import sys; from pathlib import Path; '
                'assert Path(sys.prefix).resolve()==Path(sys.argv[1]).resolve(); '
                'assert sys.prefix != sys.base_prefix',env],required=False)
            return probe.returncode==0
        except OSError:
            return False

    if not valid_python():
        # Kaggle may omit ensurepip. Create the interpreter without invoking it.
        created=command([sys.executable,'-m','venv','--without-pip',
                         '--system-site-packages',env],required=False)
        if created.returncode or not valid_python():
            # Fallback also handles installations without the stdlib venv module.
            bootstrap=WORK/'virtualenv_bootstrap'
            command([sys.executable,'-m','pip','install','--target',bootstrap,'virtualenv'])
            launcher='import sys,runpy; sys.path.insert(0,sys.argv.pop(1)); runpy.run_module("virtualenv",run_name="__main__")'
            command([sys.executable,'-c',launcher,bootstrap,'--system-site-packages',env])
        if not valid_python():
            raise RuntimeError(f'Invalid environment interpreter: {py}. See {log}')
    # Parent pip can install into a pip-less environment; no ensurepip needed.
    pip_check=command([py,'-m','pip','--version'],required=False)
    if pip_check.returncode:
        command([sys.executable,'-m','pip','--python',py,'install','pip'])
    command([py,'-m','pip','--version'])
    pins=[package+'=='+metadata.version(package) for package in ('torch','torchvision','numpy')]
    constraint=env/'constraints.txt'
    constraint.write_text(chr(10).join(pins),encoding='utf-8')
    command([py,'-m','pip','install','-c',constraint,*packages])
    frozen=command([py,'-m','pip','freeze'])
    (RUN/(name+'_pip-freeze.txt')).write_text(frozen.stdout,encoding='utf-8')
    return py

def run_worker(name, py, worker, config):
    tag='smoke' if config['smoke'] else 'full'
    config_path=RUN/(name+'_'+tag+'_config.json')
    save_json(config_path,config)
    log=RUN/(name+'_'+tag+'.log')
    with log.open('w',encoding='utf-8') as f:
        proc=subprocess.Popen([str(py),'-u',str(worker),str(config_path)],
                              stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in proc.stdout:
                print(line,end='',flush=True); f.write(line); f.flush()
            rc=proc.wait()
        finally:
            if proc.poll() is None:
                proc.terminate(); proc.wait()
            proc.stdout.close()
    if rc:
        raise RuntimeError(f'{name} {tag} failed. Log: {log}')
    return json.loads((Path(config['output'])/'summary.json').read_text(encoding='utf-8'))

print('GPU:',torch.cuda.get_device_name(0),'torch:',torch.__version__)
save_json(RUN/'environment.json',dict(python=sys.version,torch=torch.__version__,cuda=torch.version.cuda,
    gpu=torch.cuda.get_device_name(0),warmup=WARMUP,repeats=REPEATS,meters_per_unit=METERS_PER_UNIT))

## Kaggle 데이터 다운로드 (Colab CPU·디스크 사용)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'], check=True)
import kagglehub
# getpass blocks until token input completes; the login widget does not.
from getpass import getpass
from kagglehub.config import set_kaggle_api_token
_token = getpass('Kaggle API token (hidden): ').strip()
try:
    set_kaggle_api_token(_token)
finally:
    del _token
identity = kagglehub.whoami()
print('Authenticated Kaggle account:', identity['username'])
from kagglehub.exceptions import KaggleApiHTTPError
try:
    path = kagglehub.dataset_download("heeyunju/visloc-seq-eval")
except KaggleApiHTTPError as exc:
    status = getattr(getattr(exc, 'response', None), 'status_code', None)
    if status in (401, 403):
        raise RuntimeError(
            'Kaggle 접근 거부: 인증한 계정으로 '
            'https://www.kaggle.com/datasets/heeyunju/visloc-seq-eval 을 열어 '
            '다운로드 권한을 확인하세요. 비공개 데이터는 소유자의 접근 허용이 필요합니다. '
            '동의 화면이 있으면 직접 동의한 뒤 이 셀을 재실행하세요.'
        ) from exc
    raise
print("Path to dataset files:", path)
INPUT_ROOT = Path(path)
if not INPUT_ROOT.is_dir():
    raise FileNotFoundError(f'Dataset download directory missing: {INPUT_ROOT}')
save_json(RUN/'dataset_download.json', dict(handle='heeyunju/visloc-seq-eval', local_path=str(INPUT_ROOT)))

## 3. 내장 Bearing 항법 코드

In [ ]:
SOURCES = {'bearing_adapter.py': 'from pathlib import Path\nimport numpy as np\nimport torch\nfrom hashlib import sha256 as _sha256\ndef sha256(path):\n    h=_sha256()\n    with open(path,"rb") as f:\n        for b in iter(lambda:f.read(1048576),b""): h.update(b)\n    return h.hexdigest()\n\nclass BearingAdapter:\n    def __init__(self, repo, weights_dir, device):\n        import sys\n        sys.path.insert(0, str(repo))\n        # Official utils_transform accesses np.complex_ during import on NumPy 2.\n        for name, value in {\'complex_\': np.complex128, \'float_\': np.float64}.items():\n            if name not in np.__dict__:\n                setattr(np, name, value)\n        # imgaug 0.4 imports np.sctypes, removed in NumPy 2. This compatibility alias\n        # affects import-time augmentation setup; evaluation uses only ToTensor/Normalize.\n        if not hasattr(np, \'sctypes\'):\n            np.sctypes = {\'int\': [np.int8, np.int16, np.int32, np.int64],\n                          \'uint\': [np.uint8, np.uint16, np.uint32, np.uint64],\n                          \'float\': [np.float16, np.float32, np.float64],\n                          \'complex\': [np.complex64, np.complex128], \'others\': [bool, object, str, bytes]}\n        from cvphr.models.posaglreg.models import load_config_and_model\n        cls, kw = load_config_and_model(str(weights_dir))\n        self.model = cls(**kw)\n        path = Path(weights_dir) / \'best_model.pth\'\n        state = torch.load(path, map_location=\'cpu\', weights_only=True)\n        self.model.load_state_dict(state[\'model_state_dict\'], strict=True)\n        self.model = self.model.float().to(device).eval()\n        self.device, self.modules = device, [self.model]\n        self.metadata = {\'model\': \'Bearing-UAV\', \'variant\': \'official cross_view\',\n                         \'weights\': str(path), \'checkpoint_sha256\': sha256(path),\n                         \'training_regime\': \'public-pretrained: training-city overlap must be audited\'}\n\n    def predict(self, uav, block):\n        # Official p1,p2,p3,p4 order = TL,TR,BL,BR; fifth image = UAV.\n        patches = [block[y:y+256, x:x+256] for x, y in [(0, 0), (256, 0), (0, 256), (256, 256)]] + [uav]\n        x = np.stack(patches).astype(np.float32) / 255\n        x = (x - np.array([.485, .456, .406], np.float32)) / np.array([.229, .224, .225], np.float32)\n        x = torch.from_numpy(x.transpose(0, 3, 1, 2).copy())[None].to(self.device)\n        pos, _direction = self.model(x)\n        xy = pos[0].float().cpu().numpy()\n        return {\'x\': float(xy[0]), \'y\': float(xy[1]), \'valid\': True}\n', 'sequence_replay.py': '"""실제 비행 재생 항법 (UAV-VisLoc seq_eval). nav.py의 닫힌 루프 시뮬레이터와 따로 둔다.\n\n실제 사진은 드론이 날아간 경로에서만 찍혀 있어, 추정대로 드론을 움직이고 그 자리 사진을 새로 가져올 수 없다\n(nav.py nav_run의 frames). 그래서 찍힌 순서대로 사진을 주고, 매 걸음\n  추측 위치 = 직전 추정 + 이동량(실제 이동 + VIO 오차),  블록 = 추측 위치를 담은 블록,  추정 = loc(사진, 블록)\n으로 위치를 잇는다 (열린 루프). 정답 GPS는 사진 순서와 채점, 이동량 시뮬레이션에만 쓴다. 출발 위치는 정답으로 준다.\n\n모든 방법이 같은 구간·출발점·이동 오차(같은 seed)·블록 고르기 규칙을 쓰므로 짝지어 비교할 수 있다.\nloc은 nav_run과 같은 형식이다: loc(crops, headings, blocks) → (n, 2) 블록 단위 위치, 선택으로 loc.last = [{n_inl, ...}].\n그래서 기준선(B1, CorrMatchNet 등)은 3D 항법에 꽂았던 loc을 지도만 바꿔 그대로 쓴다:\n\n  from imatch.replay import run_segments, summarize\n  steps, segs = run_segments(lambda city: B1Loc(model, city), \'data/visloc/seq_eval\', odo=\'vio\', seed=2026)\n  # B1Loc: 3D 항법 때 b1_loc_sg를 만든 클래스. 구간마다 지도(city)만 바뀐다\n  print(summarize(steps, segs))\n\nnav_run과 다른 점\n- 드론은 정답 경로를 그대로 지나간다. 그래서 "도착 성공" 대신 걸음별 오차와 마지막 오차로 잰다.\n- 이동량 오차는 nav.ODO[\'vio\']와 같은 분포를 실제 이동(사진 사이 GPS 변위)에 넣는다:\n  구간마다 heading 치우침 N(0, 3°), 걸음마다 heading N(0, 2°), 걸음 길이 N(1, 5%).\n- 걸음 간격이 60~130 m다 (3D는 15 m).\n"""\nimport os\n\nimport cv2\nimport numpy as np\nimport pandas as pd\n\nN_BLOCK = 15\nPX_PER_UNIT = 128.0\nPX = 4.0\nODO = {\'none\': None, \'vio\': dict(bias_deg=3.0, head_deg=2.0, scale=0.05)}\n\n\ndef _rot(v, deg):\n    a = np.radians(deg); c, s = np.cos(a), np.sin(a)\n    return np.array([c * v[0] - s * v[1], s * v[0] + c * v[1]])\n\n\ndef load_segments(data, sets=None, max_dist=None):\n    """seq_eval/samples.csv → {set: 걸음 표(k 순서)}. max_dist(m)면 구간 시작부터 그 거리까지의 사진만."""\n    S = pd.read_csv(os.path.join(data, \'samples.csv\'))\n    if sets:\n        S = S[S.set.isin(sets)]\n    if max_dist is not None:\n        S = S[S.dist_m <= max_dist + 1e-6]\n    return {st: g.sort_values(\'k\').reset_index(drop=True) for st, g in S.groupby(\'set\', sort=True)}\n\n\ndef replay_one(loc, g, data, st, i, odo=None, seed=0, reject_inl=0, gate_m=0., recover=None, oracle_block=False,\n               n_block=N_BLOCK, log=None):\n    """구간 하나. i = 구간 번호 (이동 오차 난수를 구간마다 고정하는 데 씀). 걸음마다 log에 dict를 덧붙인다."""\n    real = g[[\'map_x\', \'map_y\']].to_numpy(float)\n    rng = np.random.default_rng([seed + 1, i])\n    n = len(g)\n    bias = rng.normal(0, odo[\'bias_deg\']) if odo else 0.0\n    dh = rng.normal(0, odo[\'head_deg\'], n) if odo else np.zeros(n)\n    ds = rng.normal(0, odo[\'scale\'], n) if odo else np.zeros(n)\n    est, streak = real[0].copy(), 0\n    for k in range(n):\n        if k == 0:\n            guess, course = real[0].copy(), 0.0\n        else:\n            d = _rot(real[k] - real[k - 1], bias + dh[k]) * (1 + ds[k])   # 드론이 아는 이동량 (지도 px, y 아래)\n            guess = est + d\n            course = float(np.degrees(np.arctan2(-d[1], d[0])))           # nav_run heading과 같은 규칙\n        ref = real[k] if oracle_block else guess\n        bx = int(np.clip(np.round((ref[0] - 256) / 256), 0, n_block - 1))\n        by = int(np.clip(np.round((ref[1] - 256) / 256), 0, n_block - 1))\n        cen = np.array([bx * 256 + 256, by * 256 + 256], float)\n        crop = cv2.cvtColor(cv2.imread(os.path.join(data, st, f\'k{int(g.k[k]):03d}_uav.jpg\')), cv2.COLOR_BGR2RGB)\n        if k == 0:  # known initial position: do not relocalize the anchor\n            pos = (real[0] - cen) / PX_PER_UNIT\n            last = {\'sec\': 0.0}\n        else:\n            pos = np.asarray(loc([crop], np.array([course]), [(bx, by)]), float)[0]\n            last = (getattr(loc, \'last\', None) or [{}])[0]\n        raw = cen + pos * PX_PER_UNIT\n        n_inl = last.get(\'n_inl\', np.inf)\n        unknown = (reject_inl > 0 and n_inl < reject_inl) or (gate_m > 0 and np.linalg.norm(raw - guess) / PX > gate_m)\n        streak = streak + 1 if unknown else 0\n        p, how = raw, \'match\'\n        if unknown:\n            q = recover(i, crop, guess, streak) if recover else None\n            p, how = (np.asarray(q, float), \'recover\') if q is not None else (guess, \'guess\')\n            if q is not None:\n                streak = 0\n        est = p\n        if log is not None:\n            log.append(dict(set=st, traj=i, k=int(g.k[k]), dist_m=float(g.dist_m[k]), real_x=real[k, 0], real_y=real[k, 1],\n                            guess_x=guess[0], guess_y=guess[1], est_x=p[0], est_y=p[1], block_x=bx, block_y=by,\n                            err_m=float(np.linalg.norm(p - real[k]) / PX), raw_err_m=float(np.linalg.norm(raw - real[k]) / PX),\n                            guess_err_m=float(np.linalg.norm(guess - real[k]) / PX),\n                            outside_block=int(np.abs(real[k] - cen).max() > 128), unknown=int(unknown), used=how,\n                            tilted=bool(g.tilted[k]) if \'tilted\' in g else False,\n                            turning=bool(g.turning[k]) if \'turning\' in g else False, **last))\n    return est\n\n\ndef run_segments(make_loc, data, sets=None, odo=\'vio\', seed=2026, reject_inl=0, gate_m=0., make_recover=None,\n                 oracle_block=False, max_dist=None, verbose=True):\n    """모든 구간을 재생한다. make_loc(city) → loc, make_recover(loc) → recover 또는 None.\n\n    반환: (걸음 표, 구간 표). 구간 표: 걸음 수, 길이, 마지막 오차, 걸음별 10/20/32 m 비율, 모름 비율.\n    """\n    odo = ODO[odo] if isinstance(odo, str) else odo\n    segs = load_segments(data, sets, max_dist)\n    steps, rows = [], []\n    for i, (st, g) in enumerate(segs.items()):\n        city = cv2.cvtColor(cv2.imread(os.path.join(data, \'maps\', f\'{st}.jpg\')), cv2.COLOR_BGR2RGB)\n        loc = make_loc(city)\n        rec = make_recover(loc) if make_recover else None\n        log = []\n        replay_one(loc, g, data, st, i, odo=odo, seed=seed, reject_inl=reject_inl, gate_m=gate_m, recover=rec,\n                   oracle_block=oracle_block, log=log)\n        if rec is not None and getattr(rec, \'log\', None):\n            L = pd.DataFrame(log)\n            R = pd.DataFrame(rec.log, index=L.index[L.unknown == 1])\n            L = L.join(R.drop(columns=\'traj\'))\n            if \'sec\' in L:\n                L[\'sec\'] += L.rec_sec.fillna(0)\n            log = L.to_dict(\'records\')\n        steps += log\n        e = np.array([s[\'err_m\'] for s in log])\n        rows.append(dict(set=st, traj=i, n_steps=len(log), length_m=float(g.dist_m.iloc[-1]), final_err_m=float(e[-1]),\n                         max_err_m=float(e.max()), med_err_m=float(np.median(e)), w10=float((e <= 10).mean()),\n                         w20=float((e <= 20).mean()), w32=float((e <= 32).mean()),\n                         unknown=float(np.mean([s[\'unknown\'] for s in log]))))\n        if verbose and (i + 1) % 10 == 0:\n            print(f\'  {i + 1}/{len(segs)} 구간\', flush=True)\n    return pd.DataFrame(steps), pd.DataFrame(rows)\n\n\ndef summarize(steps, segs):\n    """한 줄 요약 dict. 걸음 비율은 출발 걸음(k = 0, 위치를 줌)을 뺀다."""\n    s = steps[steps.k != steps.groupby(\'set\').k.transform(\'min\')]\n    return dict(segments=len(segs), steps=len(s), w10=round(float((s.err_m <= 10).mean() * 100), 1),\n                w20=round(float((s.err_m <= 20).mean() * 100), 1), w32=round(float((s.err_m <= 32).mean() * 100), 1),\n                med_err_m=round(float(s.err_m.median()), 1), final_med_m=round(float(segs.final_err_m.median()), 1),\n                final_le40=round(float((segs.final_err_m <= 40).mean() * 100), 1),\n                unknown=round(float(s.unknown.mean() * 100), 1), outside_block=round(float(s.outside_block.mean() * 100), 1))\n', 'bearing_sequence_worker.py': "\nfrom pathlib import Path\nimport sys, json, time, random\nimport cv2, numpy as np, pandas as pd, torch\nfrom bearing_adapter import BearingAdapter, sha256\nfrom sequence_replay import load_segments, replay_one, summarize, ODO\n\nclass BearingLoc:\n    def __init__(self, adapter, city): self.adapter,self.city,self.last=adapter,city,[]\n    def __call__(self, crops, headings, blocks):\n        output=[]; self.last=[]\n        for crop,(bx,by) in zip(crops,blocks):\n            block=self.city[by*256:by*256+512,bx*256:bx*256+512]\n            if crop.shape!=(256,256,3) or block.shape!=(512,512,3):\n                raise ValueError(f'Unexpected image shapes: {crop.shape}, {block.shape}')\n            if torch.cuda.is_available(): torch.cuda.synchronize()\n            t=time.perf_counter()\n            with torch.inference_mode(): pred=self.adapter.predict(crop,block)\n            if torch.cuda.is_available(): torch.cuda.synchronize()\n            xy=np.array([pred['x'],pred['y']],float)\n            if not pred['valid'] or not np.isfinite(xy).all(): raise ValueError('Invalid Bearing prediction')\n            output.append(xy); self.last.append(dict(sec=time.perf_counter()-t))\n        return np.array(output)\n\ndef main(config):\n    out=Path(config['output']); out.mkdir(parents=True,exist_ok=True)\n    if (out/'summary.json').exists(): raise RuntimeError('Output already completed; create a new RUN to rerun.')\n    random.seed(config['seed']); np.random.seed(config['seed']); torch.manual_seed(config['seed'])\n    cv2.setRNGSeed(config['seed'])\n    torch.backends.cuda.matmul.allow_tf32=False; torch.backends.cudnn.allow_tf32=False\n    torch.backends.cudnn.benchmark=False\n    adapter=BearingAdapter(Path(config['repo']),config['weights_dir'],'cuda:0')\n    if adapter.metadata['checkpoint_sha256']!=config['checkpoint_sha256']: raise ValueError('Checkpoint changed')\n    groups=load_segments(config['data'],sets=config['sets'] or None)\n    if config['smoke']:\n        st=next(iter(groups)); groups={st:groups[st].head(3).copy()}\n    logs=[]; rows=[]; t=time.perf_counter()\n    for i,(st,g) in enumerate(groups.items()):\n        city=cv2.cvtColor(cv2.imread(str(Path(config['data'])/'maps'/f'{st}.jpg')),cv2.COLOR_BGR2RGB)\n        loc=BearingLoc(adapter,city)\n        if i==0:\n            crop=cv2.cvtColor(cv2.imread(str(Path(config['data'])/st/f'k{int(g.k.iloc[0]):03d}_uav.jpg')),cv2.COLOR_BGR2RGB)\n            for _ in range(5): loc([crop],None,[(0,0)])\n        log=[]\n        replay_one(loc,g,config['data'],st,i,odo=ODO[config['odo']],seed=config['seed'],log=log)\n        L=pd.DataFrame(log); scored=L.iloc[1:]; e=scored.err_m\n        rows.append(dict(set=st,traj=i,n_steps=len(scored),length_m=float(g.dist_m.iloc[-1]),\n            final_err_m=float(e.iloc[-1]),max_err_m=float(e.max()),med_err_m=float(e.median()),\n            w10=float((e<=10).mean()),w20=float((e<=20).mean()),w32=float((e<=32).mean()),unknown=0.0))\n        logs.extend(log)\n        L.to_csv(out/'steps.csv',mode='w' if i==0 else 'a',header=i==0,index=False)\n        pd.DataFrame(rows).to_csv(out/'segments.csv',index=False)\n        print(f'{i+1}/{len(groups)} {st}: final {e.iloc[-1]:.1f} m',flush=True)\n    steps=pd.DataFrame(logs); segs=pd.DataFrame(rows); result=summarize(steps,segs)\n    scored=steps[steps.k!=steps.groupby('set').k.transform('min')]\n    result.update(smoke=config['smoke'],seed=config['seed'],odo=config['odo'],\n        known_start=True,wall_s=time.perf_counter()-t,mean_inference_s=float(scored.sec.mean()),\n        latency_scope='preprocessing + H2D + forward; excludes disk IO; warmup5',**adapter.metadata)\n    result['variant']='r0'\n    (out/'summary.json').write_text(json.dumps(result,indent=2),encoding='utf-8')\n    pd.DataFrame([result]).to_csv(out/'summary.csv',index=False)\n    print(json.dumps(result,indent=2),flush=True)\nif __name__=='__main__': main(json.loads(Path(sys.argv[1]).read_text(encoding='utf-8')))\n"}
for rel,text in SOURCES.items():
    (SOURCE/rel).write_text(text,encoding='utf-8')
sys.path.insert(0,str(SOURCE))
save_json(RUN/'source_hashes.json',{rel:hash_file(SOURCE/rel) for rel in SOURCES})

## 4. seq_eval 입력 탐색 및 전체 파일 검사

In [ ]:
import zipfile
required={'set','k','map_x','map_y','dist_m'}
def candidates(root):
    found=[]
    for p in Path(root).rglob('samples.csv'):
        if required.issubset(pd.read_csv(p,nrows=0).columns): found.append(p.parent)
    return sorted(set(found))
search=Path(DATA_DIR) if DATA_DIR else INPUT_ROOT
roots=candidates(search)
# seq_eval.zip 또는 samples.csv + maps.zip/구간별 ZIP 구조를 모두 지원
archives=list(search.rglob('*.zip'))
if archives:
    extracted=WORK/'sequence_input'; extracted.mkdir(exist_ok=True)
    relevant=[]
    for archive in archives:
        with zipfile.ZipFile(archive) as z:
            names=z.namelist()
            if any(Path(n).name=='samples.csv' or '/maps/' in '/'+n or Path(n).name.startswith('k') and n.endswith('_uav.jpg') for n in names):
                relevant.append(archive)
    for archive in relevant:
        target=extracted/str(archive.parent.relative_to(search))
        target.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(archive) as z:
            for member in z.infolist():
                if not (target/member.filename).resolve().is_relative_to(target.resolve()): raise ValueError('Unsafe ZIP path')
            z.extractall(target)
        for loose in archive.parent.iterdir():
            if loose.is_file() and loose.suffix=='.csv': shutil.copy2(loose,target/loose.name)
            elif loose.is_dir(): shutil.copytree(loose,target/loose.name,dirs_exist_ok=True)
    roots=candidates(extracted)
if len(roots)!=1: raise ValueError(f'Choose DATA_DIR containing one seq_eval dataset: {roots}')
DATA=roots[0]; table=pd.read_csv(DATA/'samples.csv')
if SETS: table=table[table['set'].isin(SETS)].copy()
if table.empty or table.duplicated(['set','k']).any(): raise ValueError('Empty or duplicate (set,k) rows')
if not np.isfinite(table[['k','map_x','map_y','dist_m']].to_numpy(float)).all(): raise ValueError('Nonfinite coordinates')
for st,g in table.groupby('set'):
    if Path(st).name!=st: raise ValueError('Invalid set name')
    g=g.sort_values('k')
    if len(g)<2 or (np.diff(g.dist_m)<0).any(): raise ValueError(f'Invalid sequence: {st}')
    city=cv2.imread(str(DATA/'maps'/f'{st}.jpg'))
    if city is None or city.shape!=(4096,4096,3): raise ValueError(f'Missing/invalid map: {st}')
    for k in g.k:
        if int(k)!=k: raise ValueError('k must be integer')
        path=DATA/st/f'k{int(k):03d}_uav.jpg'
        im=cv2.imread(str(path))
        if im is None or im.shape!=(256,256,3): raise ValueError(f'Missing/invalid UAV image: {path}')
print('DATA:',DATA,'segments:',table['set'].nunique(),'frames:',len(table))
save_json(RUN/'data_audit.json',dict(data=str(DATA),samples_sha256=hash_file(DATA/'samples.csv'),
    segments=int(table['set'].nunique()),frames=len(table),sets=SETS,seed=SEED,odo=ODO))
def nav_config(smoke):
    prepared=json.loads((RUN/'Bearing-UAV_prepared.json').read_text(encoding='utf-8'))
    return dict(**prepared,data=str(DATA),sets=SETS,seed=SEED,odo=ODO,smoke=smoke,
                output=str(RUN/'Bearing-UAV'/('smoke' if smoke else 'full')))

## 5. Bearing 가중치 준비 및 첫 구간 3장 검증

In [ ]:
BEARING_WEIGHTS_DIR = ''  # 비우면 공식 cross_view 가중치를 다운로드
BEARING_REVISION = 'main'
BEARING_PY=make_env('bearing',['huggingface_hub','imgaug==0.4.0','scikit-image','scipy'])

import zipfile
BEARING_REPO = CACHE / 'bearinguav'
BEARING_READY, BEARING_ERROR = False, ''
try:
    if not (BEARING_REPO / '.git').exists():
        subprocess.run(['git', 'clone', 'https://github.com/liukejia121/bearinguav.git', str(BEARING_REPO)], check=True)
    subprocess.run(['git', '-C', str(BEARING_REPO), 'checkout', BEARING_REVISION], check=True)
    bearing_commit = subprocess.check_output(['git', '-C', str(BEARING_REPO), 'rev-parse', 'HEAD'], text=True).strip()
    if not BEARING_WEIGHTS_DIR:
        archive = subprocess.check_output([str(BEARING_PY),'-c',
            "from huggingface_hub import hf_hub_download; print(hf_hub_download('HaoyZhou/bearinguav','Bearing_UAV.zip'))"],text=True).strip().splitlines()[-1]
        destination = CACHE / 'bearing_weights'
        destination.mkdir(exist_ok=True)
        with zipfile.ZipFile(archive) as z:
            for member in z.infolist():
                resolved = (destination / member.filename).resolve()
                if not resolved.is_relative_to(destination.resolve()):
                    raise ValueError('Unsafe ZIP path')
            z.extractall(destination)
        candidates = [p.parent for p in destination.rglob('best_model.pth') if p.parent.name == 'cross_view']
        if len(candidates) != 1:
            raise ValueError(f'Select BEARING_WEIGHTS_DIR explicitly: {candidates}')
        BEARING_WEIGHTS_DIR = str(candidates[0])
    if not (Path(BEARING_WEIGHTS_DIR) / 'training_configure.json').is_file():
        raise FileNotFoundError('training_configure.json is required alongside best_model.pth')
    BEARING_READY = True
except Exception as exc:
    BEARING_ERROR = repr(exc)
    print('Bearing-UAV preparation failed:', BEARING_ERROR)

if not BEARING_READY: raise RuntimeError(BEARING_ERROR)
save_json(RUN/'Bearing-UAV_prepared.json',dict(repo=str(BEARING_REPO),repo_commit=bearing_commit,
    weights_dir=BEARING_WEIGHTS_DIR,checkpoint_sha256=hash_file(Path(BEARING_WEIGHTS_DIR)/'best_model.pth')))
bearing_smoke=run_worker('Bearing-UAV',BEARING_PY,SOURCE/'bearing_sequence_worker.py',nav_config(True))
print('Bearing-UAV pretrained 확인 완료')

## 6. Bearing 전체 구간 항법 평가

In [ ]:
result=run_worker('Bearing-UAV',BEARING_PY,SOURCE/'bearing_sequence_worker.py',nav_config(False))
display(pd.DataFrame([result]))

## 7. 결과 표 및 다운로드

In [ ]:
p=RUN/'Bearing-UAV/full/summary.json'
if not p.exists(): raise RuntimeError('Full evaluation is not complete; inspect the full log and partial CSVs.')
summary=json.loads(p.read_text(encoding='utf-8'))
display(pd.DataFrame([summary])[['model','segments','steps','w10','w20','w32','med_err_m','final_med_m','final_le40','mean_inference_s']])
archive=shutil.make_archive(str(RUN)+'_results','zip',root_dir=RUN)
display(FileLink(archive))
print('결과 ZIP:', archive)
from google.colab import files
files.download(archive)

## 선택: 결과 ZIP을 Google Drive에도 저장
`SAVE_TO_DRIVE = True`로 바꾸고 실행하면 Drive 연결 후 결과 ZIP을 복사합니다.

In [ ]:
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    destination = Path('/content/drive/MyDrive/bearing_sequence_results')
    destination.mkdir(parents=True, exist_ok=True)
    saved = shutil.copy2(archive, destination/Path(archive).name)
    print('Drive 저장 완료:', saved)